In [ ]:
# Install required libraries
%pip install -q "datasets<=2.21.0" transformers torch scikit-learn accelerate evaluate

import os
import torch
import numpy as np
import evaluate 
import warnings
from datasets import load_dataset 
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

warnings.filterwarnings('ignore')

# Load the allagree dataset from Hugging Face Hub
dataset = load_dataset('financial_phrasebank', 'sentences_allagree', trust_remote_code=True)

# Split data into train and validation sets
dataset = dataset['train'].train_test_split(test_size=0.1, stratify_by_column="label", seed=42) # type: ignore
train_dataset = dataset['train'] 
val_dataset = dataset['test'] 

print(f"Loaded {len(train_dataset)} training sentences and {len(val_dataset)} validation sentences.")

MODEL_NAME = 'distilroberta-base' 

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# Prove to Pyrefly that tokenizer is valid
assert tokenizer is not None, "Tokenizer failed to initialize"

def tokenize_function(examples):
    return tokenizer(examples['sentence'], padding='max_length', truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True) # type: ignore
tokenized_val = val_dataset.map(tokenize_function, batched=True) # type: ignore

id2label = {0: 'negative', 1: 'neutral', 2: 'positive'}
label2id = {'negative': 0, 'neutral': 1, 'positive': 2}

# Initialize model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)
# Prove to Pyrefly that model is valid
assert model is not None, "Model failed to initialize"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device) 
print(f"Using device: {device}")

metric = evaluate.load("accuracy") 

# Explicitly type the return signature for the Trainer
def compute_metrics(eval_pred) -> dict:
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    result = metric.compute(predictions=predictions, references=labels) 
    # Ensure a dict is always returned
    return result if result is not None else {}

training_args = TrainingArguments( 
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    weight_decay=0.01,
    eval_strategy='epoch', # Updated from evaluation_strategy
    save_strategy='epoch',
    load_best_model_at_end=True
    # logging_dir removed to clear the linting error; defaults to './runs'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train, # type: ignore
    eval_dataset=tokenized_val, # type: ignore
    compute_metrics=compute_metrics
)

trainer.train()

SAVE_PATH = '../models/sentiment_model'
os.makedirs(SAVE_PATH, exist_ok=True)

model.save_pretrained(SAVE_PATH) 
tokenizer.save_pretrained(SAVE_PATH) 

print(f"Model and tokenizer successfully saved to {SAVE_PATH}")